In [ ]:
import pandas as pd

In [ ]:
df  = pd.read_csv('/banking77_full (1).csv')

In [ ]:
df.head()

,text,label,label_text,split
0,I am still waiting on my card?,11,card_arrival,train
1,What can I do if my card still hasn't arrived ...,11,card_arrival,train
2,I have been waiting over a week. Is the card s...,11,card_arrival,train
3,Can I track my card while it is in the process...,11,card_arrival,train
4,"How do I know if I will get my card, or if it ...",11,card_arrival,train


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13069 entries, 0 to 13068
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   text        13069 non-null  object
 1   label       13069 non-null  int64 
 2   label_text  13069 non-null  object
 3   split       13069 non-null  object
dtypes: int64(1), object(3)
memory usage: 408.5+ KB


In [ ]:
df.describe()

,label
count,13069.000000
mean,37.741296
std,22.355125
min,0.000000
25%,18.000000
50%,37.000000
75%,57.000000
max,76.000000


In [ ]:
df['label_text'].value_counts()

,count
label_text,
card_payment_fee_charged,227
direct_debit_payment_not_recognised,222
balance_not_updated_after_cheque_or_cash_deposit,221
wrong_amount_of_cash_received,220
cash_withdrawal_charge,217
...,...
lost_or_stolen_card,122
card_swallowed,101
card_acceptance,99


In [ ]:
df.drop(columns=["split"], errors="ignore", inplace=True)
df.drop(columns=["label"], errors="ignore", inplace=True)
df.head()

,text,label_text
0,I am still waiting on my card?,card_arrival
1,What can I do if my card still hasn't arrived ...,card_arrival
2,I have been waiting over a week. Is the card s...,card_arrival
3,Can I track my card while it is in the process...,card_arrival
4,"How do I know if I will get my card, or if it ...",card_arrival


In [ ]:
df["text"] = df["text"].str.lower()
df["text"] = df["text"].str.strip()

In [ ]:
X = df['text']
y = df['label_text']

from sklearn import model_selection
X_train, X_test, y_train, y_test = model_selection.train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
print("Total classes:", df["label_text"].nunique())
print(df["label_text"].value_counts())

Total classes: 77
label_text
card_payment_fee_charged                            227
direct_debit_payment_not_recognised                 222
balance_not_updated_after_cheque_or_cash_deposit    221
wrong_amount_of_cash_received                       220
cash_withdrawal_charge                              217
                                                   ... 
lost_or_stolen_card                                 122
card_swallowed                                      101
card_acceptance                                      99
virtual_card_not_working                             81
contactless_not_working                              75
Name: count, Length: 77, dtype: int64


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB

model = Pipeline([
    ("tfidf", TfidfVectorizer(
    max_df=0.9,
    min_df=2,
    ngram_range=(1, 1),
    sublinear_tf=True
)),
    ("nb", MultinomialNB(alpha=0.1))
])


In [ ]:
model.fit(X_train, y_train)

Pipeline(steps=[('tfidf',
                 TfidfVectorizer(max_df=0.9, min_df=2, sublinear_tf=True)),
                ('nb', MultinomialNB(alpha=0.1))])

In [ ]:
y_pred = model.predict(X_test)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, average="weighted"))
print("Recall   :", recall_score(y_test, y_pred, average="weighted"))
print("F1 Score :", f1_score(y_test, y_pred, average="weighted"))

Accuracy : 0.8706962509563887
Precision: 0.874789175850447
Recall   : 0.8706962509563887
F1 Score : 0.8701555878613215


In [ ]:
messages = [
    "My card has not arrived yet",
    "I was charged extra for cash withdrawal",
    "I forgot my pin and cannot use my card",
    "There is a transaction on my account that I did not make",
    "My contactless payment is not working",
    "Oh my card is not working",
    "This ATM machine is not work Properly",
    "Why my cash payment is not updated",
    "Someone stole my card"
]

predictions = model.predict(messages)

for msg, pred in zip(messages, predictions):
    print("Message:", msg)
    print("Predicted Intent:", pred)
    print("-" * 50)

Message: My card has not arrived yet
Predicted Intent: card_arrival
--------------------------------------------------
Message: I was charged extra for cash withdrawal
Predicted Intent: cash_withdrawal_charge
--------------------------------------------------
Message: I forgot my pin and cannot use my card
Predicted Intent: pin_blocked
--------------------------------------------------
Message: There is a transaction on my account that I did not make
Predicted Intent: direct_debit_payment_not_recognised
--------------------------------------------------
Message: My contactless payment is not working
Predicted Intent: contactless_not_working
--------------------------------------------------
Message: Oh my card is not working
Predicted Intent: card_not_working
--------------------------------------------------
Message: This ATM machine is not work Properly
Predicted Intent: pending_cash_withdrawal
--------------------------------------------------
Message: Why my cash payment is not upd

In [ ]:
import joblib

joblib.dump(model, "banking_intent_classifier.pkl")

['banking_intent_classifier.pkl']

In [ ]:
from google.colab import files

files.download("banking_intent_classifier.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>